# Advanced Architectures and Applications in Agentic AI
## CA1 Assessment: Adaptive Customer Service Harness

Student name: Maximiliano Herrera </br>
Student ID: d00296830

## Dataset Preparation

### Load and Analysis

In [1]:
import pandas as pd

df = pd.read_csv('customer-dataset.csv')
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [2]:
categories = set(df['category'])
print(categories)
print(len(categories))

{'SUBSCRIPTION', 'ORDER', 'SHIPPING', 'PAYMENT', 'INVOICE', 'CANCEL', 'REFUND', 'FEEDBACK', 'CONTACT', 'ACCOUNT', 'DELIVERY'}
11


In [3]:
df.groupby(["category"]).size().reset_index(name="count")

,category,count
0,ACCOUNT,5986
1,CANCEL,950
2,CONTACT,1999
3,DELIVERY,1994
4,FEEDBACK,1997
5,INVOICE,1999
6,ORDER,3988
7,PAYMENT,1998
8,REFUND,2992
9,SHIPPING,1970


In [4]:
df.groupby(["category", "intent"]).size().reset_index(name="count")

,category,intent,count
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995


### Filter dataset by categories and intents

In [5]:
# categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
# intents_selected = ['create_account', 'delete_account', 'edit_account',
#                     'cancel_order', 'change_order', 'place_order',
#                     'check_refund_policy', 'get_refund', 'track_refund'
#                    ]

categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
intents_selected = [
    'create_account','delete_account',  
    'cancel_order','place_order',
    'get_refund', 'track_refund'
]

df_filtered = df[
    df['category'].isin(categories_selected) &
    df['intent'].isin(intents_selected)
].rename(columns={"Unnamed: 0": "id"})

print(f"df_filtered: {len(df_filtered)}, {type(df_filtered)}")

df_filtered: 5983, <class 'pandas.DataFrame'>


In [6]:
import plotly.express as px

def plot_number_records_per_intent(dataframe):
    fig = px.bar(
        dataframe['intent'].value_counts().reset_index(),
        x='intent',
        y='count',
        title='Number of Records per Intent',
        text_auto=True
    )

    fig.show()

plot_number_records_per_intent(df_filtered)

In [7]:
from pandas import DataFrame

def get_split_dataset(categories_selected, intents_selected, max_total_rows=60, random_state=42):
    intent_count = len(intents_selected)
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent = (max_total_rows // intent_count) // 2 * 2
    rows_per_split = rows_per_intent // 2
    if rows_per_split < 1:
        raise ValueError("max_total_rows is too small for one train and one test row per intent")

    print(f"intents: {intent_count}")
    print(f"rows_per_intent: {rows_per_intent}")
    print(f"rows_per_split: {rows_per_split}")

    ds_filtered = df[
        df["category"].isin(categories_selected) &
        df["intent"].isin(intents_selected)
    ].rename(columns={"Unnamed: 0": "id"})
    print(f"ds_filtered: {len(ds_filtered)}")

    sampled = ds_filtered.groupby("intent", group_keys=False).sample(
        n=rows_per_intent, random_state=random_state
    )
    train_df = sampled.groupby("intent", group_keys=False).head(rows_per_split)
    test_df = sampled.drop(train_df.index)

    print(f"sampled: {len(sampled)}")
    print(f"train_df: {len(train_df)}")
    print(f"test_df: {len(test_df)}")
    return train_df, test_df

def get_dataset_batches_unbalanced(dataframe: DataFrame, batch_size=30, random_state=42):
    shuffled_df = (
        dataframe
        .drop_duplicates(subset=["instruction"]) # remove duplicated instructions
        .sample(frac=1, random_state=random_state) # shuffle records and use random_state for reproducibility
        .reset_index(drop=True) # reset index and do not insert it as new column
    )

    for start in range(0, len(shuffled_df), batch_size): #get a sequence from start (0, inclusive) to stop (len(shuffled_df), exclusive) with a step (batch_size)
        batch = shuffled_df.iloc[start:start + batch_size] # select batch_size rows starting from start

        #if batch is incomplete then break the loop
        if len(batch) < batch_size: 
            break

        # return a copy of the batch so original data is unchanged
        yield batch.copy()


def get_dataset_batches_balanced(dataframe: DataFrame, batch_size=30, random_state=42):    
    remaining = dataframe.drop_duplicates(subset=["instruction"]).copy()    
    num_intents = remaining["intent"].nunique()  
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent =  batch_size// num_intents

    if rows_per_intent == 0:
        raise ValueError("batch_size must be at least the number of intents")

    num_batches = remaining.groupby("intent").size().min() # rows per intent
    for _ in range(num_batches):
        batch = remaining.groupby("intent").sample(
            n=rows_per_intent,
            random_state=random_state
        )

        remaining= remaining.drop(batch.index)

        yield batch

def plot_and_preview_dataset(batch: DataFrame):
    plot_number_records_per_intent(batch)
    print(f"Num rows: {len(batch)}, num intents: {batch["intent"].nunique()}")
    print(len(batch))        
         

In [8]:
batches = get_dataset_batches_balanced(df_filtered,32)
train1_df =next(batches)
train2_df =next(batches)
test1_df =next(batches)
test2_df =next(batches)

TOP_N = 0

if TOP_N:
    train1_df =train1_df.head(TOP_N)
    train2_df =train2_df.head(TOP_N)
    test1_df =test1_df.head(TOP_N)
    test2_df =test2_df.head(TOP_N)

# train_df, test_df = get_split_dataset(categories_selected, intents_selected, 50)

In [9]:
plot_and_preview_dataset(train1_df)
train1_df

Num rows: 30, num intents: 6
30


,id,flags,instruction,category,intent,response
860,860,BILZ,can you help me camceling the purchase I made?,ORDER,cancel_order,I've been informed that you need assistance wi...
79,79,BIQZ,whag do i need to do to cancel order {{Order N...,ORDER,cancel_order,I understand your need for assistance in cance...
479,479,BC,"I bought some product, I want to cancel order ...",ORDER,cancel_order,I've come to understand that you've made a pur...
209,209,BCELN,"I do not want this item, cancel order {{Order ...",ORDER,cancel_order,I've observed that you no longer want the item...
527,527,BCLN,I can no longer afford purchase {{Order Number...,ORDER,cancel_order,I completely understand the importance of canc...
10658,10658,BL,I need assistance to create a {{Account Catego...,ACCOUNT,create_account,Thank you for choosing to create a {{Account C...
10677,10677,BCLN,"I don't have a {{Account Category}} account, h...",ACCOUNT,create_account,Thank you so much for expressing your interest...
10581,10581,BCIL,"I ain't registered yet, how can I open a stand...",ACCOUNT,create_account,Thank you for expressing your interest in open...
9962,9962,BILQ,can ya help me to create a platinum account fo...,ACCOUNT,create_account,Indeed! I'm thrilled to assist you in creating...
10669,10669,BKLZ,opdn standard account,ACCOUNT,create_account,For sure! I'm here to guide you through the pr...


In [10]:
plot_and_preview_dataset(train2_df)
train2_df

Num rows: 30, num intents: 6
30


,id,flags,instruction,category,intent,response
78,78,BCL,I have got to cancel purchase {{Order Number}}...,ORDER,cancel_order,I've grasped that you need assistance with can...
321,321,BCILNPQ,i dont want purchase {{Order Number}} could i ...,ORDER,cancel_order,I grasp your desire to cancel your purchase wi...
801,801,BILZ,can you help me cancelling purchase {{Order N...,ORDER,cancel_order,I'll take care of it! I'm here to assist you w...
87,87,B,I need help cancelling order {{Order Number}},ORDER,cancel_order,I can relate to the understanding that you nee...
721,721,B,help canceling order {{Order Number}},ORDER,cancel_order,I catch on to the fact that you need assistanc...
10694,10694,BILQ,can i create a {{Account Category}} account fo...,ACCOUNT,create_account,I'm happy to help! It's wonderful to hear that...
10907,10907,BLZ,I have a problem with opening a standard account,ACCOUNT,create_account,I'm sorry to hear that you're experiencing dif...
10278,10278,BL,I need assistance opening a {{Account Category...,ACCOUNT,create_account,We appreciate your inquiry to us for assistanc...
10542,10542,BQ,i need help to create a platinum account,ACCOUNT,create_account,I'd be delighted to assist you in creating a {...
10343,10343,BLZ,I need assistance with opening a {{Account Typ...,ACCOUNT,create_account,"Wow, a {{Account Type}} account! That's an exc..."


In [11]:
plot_number_records_per_intent(test1_df)
test1_df

,id,flags,instruction,category,intent,response
758,758,BQZ,i want assisgance cancelling order {{Order Num...,ORDER,cancel_order,I've taken note that you're seeking assistance...
963,963,B,problem with canceling order {{Order Number}},ORDER,cancel_order,I realized that you're encountering difficulti...
222,222,BL,I want help cancelling purchase {{Order Number}},ORDER,cancel_order,"I'm on the same page, your desire for assistan..."
863,863,BLQ,need assistance to cancel purchase {{Order Num...,ORDER,cancel_order,I acknowledge that you're seeking assistance w...
459,459,BEPQZ,i do not know how i could casncel order {{Orde...,ORDER,cancel_order,I'm conscious of the reality that you're unsur...
10696,10696,BKLZ,creatingnew {{Account Category}} account,ACCOUNT,create_account,That's great news! I'm thrilled to assist you ...
10577,10577,BILQ,how to open a {{Account Type}} account for my ...,ACCOUNT,create_account,That's fantastic! I'm thrilled to assist you i...
10279,10279,BKL,new {{Account Type}} account for wife,ACCOUNT,create_account,How exciting to hear that you want to open a n...
10543,10543,BIL,how can I get information about opening a stan...,ACCOUNT,create_account,Thank you for expressing interest in opening a...
10345,10345,BCELNQZ,i do nothave a {{Account Type}} account help s...,ACCOUNT,create_account,Appreciate you reaching out to us! I'm excited...


In [12]:
plot_number_records_per_intent(test2_df)
test2_df

,id,flags,instruction,category,intent,response
964,964,BQ,need help cancelling order {{Order Number}},ORDER,cancel_order,I've become conscious that you're seeking assi...
34,34,BQ,need assistance to cancel order {{Order Number}},ORDER,cancel_order,I've got that you're seeking assistance in can...
158,158,BILQ,where do i cancel purchase {{Order Number}},ORDER,cancel_order,I'm cognizant of the fact that you're looking ...
80,80,B,I want assistance canceling order {{Order Numb...,ORDER,cancel_order,I've realized that you're seeking assistance w...
89,89,BLQ,i want help to cancel purchase {{Order Number}},ORDER,cancel_order,I completely understand that you need assistan...
10430,10430,BLM,find information about opening {{Account Categ...,ACCOUNT,create_account,I appreciate your interest in gathering inform...
10552,10552,BKLZ,new gold acount for wife,ACCOUNT,create_account,Thank you for considering opening a new {{Acco...
10505,10505,BLM,I have problems with opening my {{Account Type...,ACCOUNT,create_account,I apologize for the inconvenience you have fac...
10726,10726,BILPQZ,would it be possible to oen a gold account,ACCOUNT,create_account,I'm delighted that you're considering opening ...
10531,10531,BKL,create new pro account for daughter,ACCOUNT,create_account,That's wonderful! I'm thrilled to assist you i...


## Test Harnes Agent

In [13]:
from general import clean_agentic_memory

clean_agentic_memory()

### Test inference mode, wihtout agentic memory

In [14]:
import logging
from general import initialize_logging
from harness_agent.harness import AgentHarness
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
# model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-astra")
# model = get_llm("gpt-6-luna-low")
model = get_llm("deepseek-flash-low")
 
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=False, agentic_memory_disabled=True)
# # response = harness.invoke("see the available payment options")
# # response = harness.invoke("were can i cancel purchase {{Order Number}}")
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-02 22:40:53 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
2026-11-02 22:40:54 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:40:56 - harness.py - invoke - INFO - Harness Response: I can help guide you, but I’m not able to directly access or modify your purchase from this chat.

To cancel it, please have these ready:
- Order number/confirmation number
- Email used for the purchase
- Item purchased and purchase date
- Whether it has shipped yet

General steps:
1. Open the order confirmation email and look for a **Cancel order** link.
2. Sign in to the store account and go to **Orders/Order History**.
3. If cancellation isn’t available, contact the seller’s support immediately with your order number and say: *“Please cancel order #[number] placed on [date]. If it has shipped, please advise on refusal/return options.”*

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='571cc554-3383-4b34-a668-8c5eae589c1d'),
  AIMessage(content='I can help guide you, but I’m not able to directly access or modify your purchase from this chat.\n\nTo cancel it, please have these ready:\n- Order number/confirmation number\n- Email used for the purchase\n- Item purchased and purchase date\n- Whether it has shipped yet\n\nGeneral steps:\n1. Open the order confirmation email and look for a **Cancel order** link.\n2. Sign in to the store account and go to **Orders/Order History**.\n3. If cancellation isn’t available, contact the seller’s support immediately with your order number and say: *“Please cancel order #[number] placed on [date]. If it has shipped, please advise on refusal/return options.”*\n4. If it already shipped, you may need to refuse delivery or request a return.\n\nIf you share the store name and order status—**not your full card

### Test multiturn

In [15]:
response = harness.invoke("Unfortunatelly I cannot give you none of those details...")
response

2026-11-02 22:40:57 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:01 - harness.py - invoke - INFO - Harness Response: No problem—you don’t need to share order details here. I still can’t cancel it directly, but you can usually get it cancelled without an order number.

Try this:

1. **Search your email** for “order confirmation,” “receipt,” or the store name. Check spam/promotions.
2. **Check your bank/card app** for the merchant name and purchase date.
3. **Log in to the store account** if you have one and look under Orders/Order History.
4. **Contact the seller’s support** by chat/email/phone and give them whatever you *do* know—your name, email used, shipping address, approximate date, item, and amount. They can often find the order from that.

You can send this message:

> Hello,  
> I’d like to cancel a recent purchase. I don’t have the order number available, but the order was placed

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='571cc554-3383-4b34-a668-8c5eae589c1d'),
  AIMessage(content='I can help guide you, but I’m not able to directly access or modify your purchase from this chat.\n\nTo cancel it, please have these ready:\n- Order number/confirmation number\n- Email used for the purchase\n- Item purchased and purchase date\n- Whether it has shipped yet\n\nGeneral steps:\n1. Open the order confirmation email and look for a **Cancel order** link.\n2. Sign in to the store account and go to **Orders/Order History**.\n3. If cancellation isn’t available, contact the seller’s support immediately with your order number and say: *“Please cancel order #[number] placed on [date]. If it has shipped, please advise on refusal/return options.”*\n4. If it already shipped, you may need to refuse delivery or request a return.\n\nIf you share the store name and order status—**not your full card

In [16]:
response = harness.invoke("Can you remind me my first question please?")
response

2026-11-02 22:41:01 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:02 - harness.py - invoke - INFO - Harness Response: Your first question was:

**“can you help me camceling the purchase I made?”**


{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='571cc554-3383-4b34-a668-8c5eae589c1d'),
  AIMessage(content='I can help guide you, but I’m not able to directly access or modify your purchase from this chat.\n\nTo cancel it, please have these ready:\n- Order number/confirmation number\n- Email used for the purchase\n- Item purchased and purchase date\n- Whether it has shipped yet\n\nGeneral steps:\n1. Open the order confirmation email and look for a **Cancel order** link.\n2. Sign in to the store account and go to **Orders/Order History**.\n3. If cancellation isn’t available, contact the seller’s support immediately with your order number and say: *“Please cancel order #[number] placed on [date]. If it has shipped, please advise on refusal/return options.”*\n4. If it already shipped, you may need to refuse delivery or request a return.\n\nIf you share the store name and order status—**not your full card

### Test training mode and enable agentic memory

In [17]:
from harness_agent.harness import AgentHarness
import logging
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-luna")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=True, agentic_memory_disabled=False)
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-02 22:41:02 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
2026-11-02 22:41:02 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:02 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-02 22:41:03 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:03 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:04 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:04 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-02 22:41:05 - _client.py - _send_single_

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='61f012f0-f68b-42b8-bca8-68890365ce3f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants help canceling a purchase. No skills listed. So we need to request supervisor for example response.', 'tool_calls': [{'id': 'fc_38346ced-7bc6-40ce-b34b-372f1c132d51', 'function': {'arguments': '{"user_query":"can you help me camceling the purchase I made?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 1021, 'total_tokens': 1084, 'completion_time': 0.17711947, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.000156698, 'prompt_tokens_details': {'cached_tokens': 509}, 'queue_time': 0.048742681, 'total_time': 0.177276168}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'fini

### Test that multiturn should be disable on training mode

In [18]:
response = harness.invoke("What was my first question?")
response

2026-11-02 22:41:06 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:06 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-02 22:41:06 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:07 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:07 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:07 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-02 22:41:09 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 22:41:09 - _client.py - _

{'messages': [HumanMessage(content='What was my first question?', additional_kwargs={}, response_metadata={}, id='612ae739-333b-4bcf-822c-772bbc63a60e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What was my first question?" This is a meta question about prior conversation. There\'s no skill matching. The skill list only has cancel-purchase-assistance. This is not about canceling purchase. So no skill matches. According to instructions, if no skill matches, call request_supervisor before answering. So we need to call request_supervisor with user_query.', 'tool_calls': [{'id': 'fc_b7efe493-6b20-4990-9997-fc2ce9df2a8b', 'function': {'arguments': '{"user_query":"What was my first question?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 713, 'total_tokens': 824, 'completion_time': 0.234302638, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0

## Testing and Evaluation

In [19]:
clean_agentic_memory()

2026-11-02 22:41:09 - general.py - clean_agentic_memory - INFO - Cleaning agentic memory...
2026-11-02 22:41:09 - general.py - clean_agentic_memory - INFO -  * rmtree Skills...
2026-11-02 22:41:09 - general.py - clean_agentic_memory - INFO -  * .skills folder created...
2026-11-02 22:41:09 - general.py - clean_agentic_memory - INFO -  * Deleted AGENT.md...
2026-11-02 22:41:09 - general.py - clean_agentic_memory - INFO -  * ./AGENT.md file created...


In [20]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError

delete_if_exists = True

client = Client()

def create_dataset(name, description, delete_if_exists= False):
    try:
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                return dataset

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        return dataset       
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = 'CA1 - Customer Dataset - v8'
dataset = create_dataset(
    name = dataset_name,
    description='Customer questions and responses',
    delete_if_exists= delete_if_exists
)

Dataset 'CA1 - Customer Dataset - v8' already exists. Deleting existing dataset...
Dataset 'CA1 - Customer Dataset - v8' created successfully.


In [21]:
from pandas import DataFrame

def create_examples(dataset: Dataset, df: DataFrame, split_name: str = 'train'):
    print(f"Creating {len(df)} examples for split: {split_name}")

    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "category": row["category"],
                "intent": row["intent"],
                "instruction": row["instruction"],
            },
            outputs= {
                "response": row["response"]                         
            }
        )

        example_ids.append(example.id)

    print(f"Set split name: {split_name}")
    client.update_examples(example_ids=example_ids,splits=[split_name] * len(example_ids))

In [22]:
SPLIT_TRAIN_1 = 'train_1'
SPLIT_TRAIN_2 = 'train_2'
SPLIT_TEST_1 = 'test_1'
SPLIT_TEST_2 = 'test_2'
create_examples(dataset, train1_df, SPLIT_TRAIN_1)
create_examples(dataset, train2_df, SPLIT_TRAIN_2)
create_examples(dataset, test1_df, SPLIT_TEST_1)
create_examples(dataset, test2_df, SPLIT_TEST_2)

Creating 30 examples for split: train_1
Set split name: train_1
Creating 30 examples for split: train_2
Set split name: train_2
Creating 30 examples for split: test_1
Set split name: test_1
Creating 30 examples for split: test_2
Set split name: test_2


In [23]:
import pandas as pd

def get_examples_by_split(dataset_name, split_name):    
    examples = list(
        client.list_examples(
            dataset_name=dataset_name,
            splits=[split_name]
        )
    )

    df_examples = pd.DataFrame([
        {
            **example.inputs, #unpack operator
            **example.outputs,
            "langsmith_id": str(example.id),
            "splits": example.metadata.get("dataset_split")
                    if example.metadata else None
        }
        for example in examples
    ])

    return {
        'name' : split_name,
        'examples' : examples,
        'df': df_examples
    }

split_train_1 = get_examples_by_split(dataset_name,SPLIT_TRAIN_1)
split_train_2 = get_examples_by_split(dataset_name,SPLIT_TRAIN_2)
split_test_1 = get_examples_by_split(dataset_name,SPLIT_TEST_1)
split_test_2 = get_examples_by_split(dataset_name,SPLIT_TEST_2)

In [24]:
from harness_agent.harness import AgentHarness
from pydantic import BaseModel, Field
from llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

class ResponseEvaluatorSchema(BaseModel):
    """Class to define the schema for quality evaluation."""
    score: int = Field(description="An integer score from 1 to 10 indicating the quality of the generated answer.")
    comment: str = Field(description="A brief explanation of the score")

def evaluator_agent_response_quality(inputs: dict, outputs: dict, reference_outputs: dict, model_name:str) -> dict:
    """Evaluate the quality of the agent's generated response using LLM with a structured output."""  
    query = inputs["instruction"]
    agent_response = outputs['agent_response']
    reference_response = reference_outputs['response']

    system_eval_prompt = "You score customer service answers."
    eval_prompt = f"""Score the agent answer from 1 to 10 against the reference.

1 means it does not match.
5 means some of it matches, but important parts are missing.
10 means it is the same kind of answer and the important information is there.

Do not give a high score just because it is polite.
If the reference has steps or placeholders, the answer should have them too.
If the answer makes up a procedure the reference does not have, score it lower.
Write a short comment on what matched and what was missing.

User query:
{query}

Reference answer:
{reference_response}

Agent response:
{agent_response}"""

    model = get_llm(model_name)
    result = model.with_structured_output(ResponseEvaluatorSchema).invoke([SystemMessage(system_eval_prompt), HumanMessage(eval_prompt)])
    
    evaluator_result = {
        "key": "response_quality",
        "score": result.score, 
        "comment": result.comment
        }
    logging.debug(f"evaluator_agent_response_quality: {evaluator_result}")

    return evaluator_result

def run_agent(query:str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool):      
    model = get_llm(model_name)
    harness = AgentHarness(model = model, training_dataset=training_dataset, training_mode=training_mode, agentic_memory_disabled=agentic_memory_disabled)
    # response = harness.invoke("see the available payment options")
    response = harness.invoke(query)
    agent_response = response['text']
    
    logging.debug("---------------------run_agent------------------------")
    logging.debug(f"response: {agent_response}")
    
    return agent_response

def target_function(inputs: str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool) -> dict:
    agent_response = run_agent(inputs["instruction"], model_name, training_dataset, training_mode, agentic_memory_disabled)
    logging.debug("---------------------target_function------------------------")
    logging.debug(f"agent_response: {agent_response}")
    
    return {"agent_response": agent_response}


In [25]:
def evaluate_agent(
    agent_model_name: str,
    judge_model_name: str,
    training: bool,
    disable_agentic_memory: bool,
    split: dict,
    stage: str,    
    run_evaluators: bool = True,
    ):
    
    #avoid answers leakage during test
    dataframe = split['df'] if training else None
    examples = split['examples']
    training_mode = "training" if training else "testing"
    agentic_memory_mode  = "memory_disabled" if disable_agentic_memory else "memory_on"
    agent_model_name_sanitized = agent_model_name.replace("/", "-")
    
    logging.debug(f"Evaluate Agent --> mode: {training_mode}, agentic_memory_mode: {agentic_memory_mode}")
    
    evaluators = []    
    if run_evaluators: 
         evaluators=[lambda inputs, outputs, reference_outputs : evaluator_agent_response_quality( inputs, outputs, reference_outputs, judge_model_name )]
         
    results =  client.evaluate(
        lambda inputs: target_function(inputs, agent_model_name, dataframe, training, disable_agentic_memory),
        data= examples,
        evaluators= evaluators,
        experiment_prefix=f"{stage}-{agent_model_name_sanitized}",       
         metadata={
            "stage": stage,           
            "model_name": agent_model_name,
            "judge_model": judge_model_name,
            "training_mode": training_mode,
            "agentic_memory": agentic_memory_mode,
            "split": split["name"],
            "n_examples": len(examples)
        },
        #Training must run sequentially because there are read/write operations into files
        max_concurrency= 1 if training else 5 ,
        #Ensure that evaluate blocks until evaluation completes
        blocking=True, 
    )
    
    return results
    
# I will not traion the agent in this way (off-line from LangSmith) because the experiments will give me more obvservability on agent's behavior
# def train_agent(model_name, split):
#     dataframe = split['df']
    
#     for row in dataframe.itertuples():
#         agent_response = run_agent(query= row["instruction"], model_name = model_name, training_dataset= dataframe, training_mode =True  , agentic_memory_disabled=False)
    

In [26]:
from general import clean_agentic_memory

def evaluation_pipeline(agent_models, judge_model, run_evaluators_on_training = False):
    results_by_model = {}
    
    for model in agent_models:
        logging.info(f"Executing evaluation pipeline for model: {model}...")   
        clean_agentic_memory()   
        model_results = {}
        #Baseline: Evaluate agent on inference mode with agentic memory disabled
        model_results["baseline"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=True, split=split_test_1, stage="baseline")
        
        #Training 1: Evaluate agent on training mode with agentic memory enabled        
        model_results["train_1"] = evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_1, stage="train_1", run_evaluators=run_evaluators_on_training)
        
        #Test 1 (after training 1): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        model_results["post_train_1"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1, stage="post_train_1")

        #Training 2: Evaluate agent on training mode with agentic memory enabled        
        model_results["train_2"] = evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_2, stage="train_2", run_evaluators=run_evaluators_on_training)
        
        #Test 2 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        model_results["post_train_2"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1, stage="post_train_2")
        
        #Test 3 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_2
        model_results["final_test"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_2, stage="final_test")
        
        results_by_model[model]=model_results
        
    return results_by_model

In [27]:
initialize_logging(logging.WARNING)

AGENT_MODELS = [
    "openai/gpt-oss-120b",
    "deepseek-flash-low",
    "gpt-6-luna-low",
    "gpt-6-luna-medium",
]

JUDGE_MODEL = "gpt-5-nano" # "deepseek-v4-pro" failing on parsing structured output
JUDGE_MODEL = "gpt-5.5" # better results
JUDGE_MODEL = "gpt-6-luna-low" # more critic
# JUDGE_MODEL = "gpt-6-luna-medium"  # more critic but slower
results = evaluation_pipeline(agent_models=AGENT_MODELS,judge_model= JUDGE_MODEL,run_evaluators_on_training= True)

View the evaluation results for experiment: 'baseline-openai-gpt-oss-120b-96bf5c91' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/57f07685-e3e9-45b0-952a-b682825b1ec6/compare?selectedSessions=37fe9918-9677-4d6f-b561-884b31e5892b


View the evaluation results for experiment: 'train_1-openai-gpt-oss-120b-31f173b2' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/57f07685-e3e9-45b0-952a-b682825b1ec6/compare?selectedSessions=575e8fd4-5ab3-41b5-9a22-9256365802c5


View the evaluation results for experiment: 'post_train_1-openai-gpt-oss-120b-c8049fd6' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/57f07685-e3e9-45b0-952a-b682825b1ec6/compare?selectedSessions=9b9a2ef4-8cca-4966-a02e-df1924380f4a


View the evaluation results for experiment: 'train_2-openai-gpt-oss-120b-1ffb50f7' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/57f07685-e3e9-45b0-952a-b68

In [28]:
results

{'openai/gpt-oss-120b': {'baseline': <ExperimentResults baseline-openai-gpt-oss-120b-96bf5c91>,
  'train_1': <ExperimentResults train_1-openai-gpt-oss-120b-31f173b2>,
  'post_train_1': <ExperimentResults post_train_1-openai-gpt-oss-120b-c8049fd6>,
  'train_2': <ExperimentResults train_2-openai-gpt-oss-120b-1ffb50f7>,
  'post_train_2': <ExperimentResults post_train_2-openai-gpt-oss-120b-3706ac15>,
  'final_test': <ExperimentResults final_test-openai-gpt-oss-120b-d3b567d9>},
 'deepseek-flash-low': {'baseline': <ExperimentResults baseline-deepseek-flash-low-f390604a>,
  'train_1': <ExperimentResults train_1-deepseek-flash-low-9ab973c8>,
  'post_train_1': <ExperimentResults post_train_1-deepseek-flash-low-ae5b420e>,
  'train_2': <ExperimentResults train_2-deepseek-flash-low-ff6a1412>,
  'post_train_2': <ExperimentResults post_train_2-deepseek-flash-low-41eaf747>,
  'final_test': <ExperimentResults final_test-deepseek-flash-low-fff40548>},
 'gpt-6-luna-low': {'baseline': <ExperimentResults 

In [29]:
df_results = pd.DataFrame(results)
df_results

,openai/gpt-oss-120b,deepseek-flash-low,gpt-6-luna-low,gpt-6-luna-medium
baseline,({'run': id=UUID('01a19e6c-b15c-73f3-ba83-783c...,({'run': id=UUID('01a19e71-cdb4-7e92-80ac-839f...,({'run': id=UUID('01a19e7d-0bf9-78c2-b94b-df7c...,({'run': id=UUID('01a19e89-4851-7160-9f5a-6d88...
train_1,({'run': id=UUID('01a19e6c-e99d-72a1-b45d-9f7a...,({'run': id=UUID('01a19e72-2dcf-76e0-ba2c-2bd5...,({'run': id=UUID('01a19e7d-3e99-7640-b4cc-ed0e...,({'run': id=UUID('01a19e89-8e8e-7c11-8c09-1037...
post_train_1,({'run': id=UUID('01a19e6f-17d0-7e01-a0f8-78dc...,({'run': id=UUID('01a19e76-bd30-7622-907d-1847...,({'run': id=UUID('01a19e82-f6b0-7d61-9cd8-04ff...,({'run': id=UUID('01a19e8f-6399-7582-9ca5-528e...
train_2,({'run': id=UUID('01a19e6f-6165-74d1-8919-e355...,({'run': id=UUID('01a19e77-1e3c-7020-be66-1f39...,({'run': id=UUID('01a19e83-5384-71b0-b265-2b48...,({'run': id=UUID('01a19e8f-d8c6-7ea1-8196-9969...
post_train_2,({'run': id=UUID('01a19e71-5d2d-7f42-91e9-90b7...,({'run': id=UUID('01a19e7c-4111-7ce1-8fa7-7d98...,({'run': id=UUID('01a19e88-7d59-7383-b992-6655...,({'run': id=UUID('01a19e97-0393-7ef1-84a4-9c4d...
final_test,({'run': id=UUID('01a19e71-96ba-74c3-92a8-6ae2...,({'run': id=UUID('01a19e7c-a56d-74d0-ba04-96c5...,({'run': id=UUID('01a19e88-e751-7c91-9ac4-9369...,({'run': id=UUID('01a19e97-74c5-71c2-970d-a2b6...
